# SMA cross-species meta-analysis — master Colab runner

This notebook runs the rebuilt workflow from audit to publication outputs. It does
not use the discarded conserved-gene or chunked-network results. Read the printed
audit messages: missing or ineligible evidence is surfaced rather than silently
filled in.

**Before running:** download the complete project ZIP. The next cell will ask you
to upload it if the project is not already present in Colab. Start with light
downloads. Raw-read reprocessing is a separate confirmatory profile documented in
`docs/RAW_REPROCESSING.md`.

In [ ]:
from pathlib import Path
import zipfile

# Colab-safe setup: locate an existing copy or request the complete project ZIP.
PREFERRED_PROJECT_ROOT = Path('/content/SMA_CrossSpecies_MetaAnalysis')
AUTO_UPLOAD_PROJECT_ZIP = True

def locate_project(search_root=Path('/content')):
    candidates = []
    preferred = PREFERRED_PROJECT_ROOT / 'pyproject.toml'
    if preferred.exists():
        candidates.append(preferred.parent)
    for pyproject in search_root.rglob('pyproject.toml'):
        try:
            if 'name = "sma-xspecies"' in pyproject.read_text(encoding='utf-8'):
                candidates.append(pyproject.parent)
        except (OSError, UnicodeDecodeError):
            pass
    unique = sorted(set(candidates), key=lambda path: (len(path.parts), str(path)))
    return unique[0] if unique else None

PROJECT_ROOT = locate_project()

if PROJECT_ROOT is None and AUTO_UPLOAD_PROJECT_ZIP:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'Project folder is missing. Unzip SMA_CrossSpecies_MetaAnalysis_v0.1.1.zip '
            'and set PREFERRED_PROJECT_ROOT to that folder.'
        ) from exc

    print('Upload the COMPLETE project ZIP (not only 00_Master_Run.ipynb).')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError(f'Upload exactly one project ZIP; received: {list(uploaded)}')

    zip_path = Path('/content') / Path(zip_names[0]).name
    if not zip_path.exists():
        zip_path.write_bytes(uploaded[zip_names[0]])

    extraction_root = Path('/content').resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            destination = (extraction_root / member.filename).resolve()
            if destination != extraction_root and extraction_root not in destination.parents:
                raise ValueError(f'Unsafe path in ZIP: {member.filename}')
        archive.extractall(extraction_root)

    PROJECT_ROOT = locate_project()

if PROJECT_ROOT is None or not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(
        'The SMA project could not be located. Upload the complete project ZIP, '
        'not the notebook file by itself.'
    )

RUN_DOWNLOADS = True
INCLUDE_LARGE_ARCHIVES = False     # GSE207890/GSE208629 raw archives are >200 MB
INSTALL_R_PACKAGES = True
FETCH_ORTHOLOGS = True
RUN_GSE209926_DE = True
RUN_GSE207890_BLOCKED_DE = True
RUN_META_ANALYSIS = True
RUN_TREATMENT_REVERSAL = True
RUN_SPLICING = True

print('Project ready:', PROJECT_ROOT)

## 1. Install and validate the locked design

In [ ]:
import os, subprocess, sys
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[enrichment,notebook,test]'], check=True)

def run(*parts, check=True):
    command = [str(part) for part in parts]
    print('$', ' '.join(command))
    return subprocess.run(command, check=check)

run('sma-xspecies', 'validate')

The validation report must contain no errors. Informational restrictions
for GSE69175 and GSE208629 are expected and are part of the design.

## 2. Retrieve small public supplementary files and record checksums

In [ ]:
if RUN_DOWNLOADS:
    command = ['sma-xspecies', 'download']
    if INCLUDE_LARGE_ARCHIVES:
        command.append('--include-large')
    run(*command, check=False)
else:
    print('Download stage disabled')

## 3. Standardize published study-level summaries

In [ ]:
run('sma-xspecies', 'standardize', check=False)
from pathlib import Path
print('\nCurrent DE inputs:')
for path in sorted((PROJECT_ROOT / 'results/de').glob('*.csv')):
    print(' ', path.name)

## 4. Install Bioconductor and pin one-to-one orthology

In [ ]:
if INSTALL_R_PACKAGES:
    run('Rscript', 'environment/install_bioc.R')
if FETCH_ORTHOLOGS:
    run('Rscript', 'R/fetch_orthologs.R', 'resources/ensembl_human_mouse_one_to_one.tsv', '113')

## 4b. Fit any processed expression matrices discovered during standardization

In [ ]:
processed_models = [
    ('GSE56284', 'Mus_musculus', 'motor_neuron', 'eligible'),
    ('GSE69175', 'Homo_sapiens', 'motor_neuron', 'sensitivity_only'),
]
for accession, species, tissue, eligibility in processed_models:
    matrix = PROJECT_ROOT / f'data/processed/{accession}_expression.tsv'
    output = PROJECT_ROOT / f'results/de/{accession}_disease-control.csv'
    if matrix.exists() and not output.exists():
        run('Rscript', 'R/run_processed_limma.R', matrix,
            f'config/sample_annotations/{accession}.csv', accession, species,
            tissue, eligibility, output)
    else:
        print(accession, ': published summary already standardized or matrix absent')

## 5. GSE209926 spinal-cord DE from sample-level counts

In [ ]:
archive = PROJECT_ROOT / 'data/raw/GSE209926/GSE209926_RAW.tar'
counts = PROJECT_ROOT / 'data/processed/GSE209926_counts.tsv'
if RUN_GSE209926_DE and archive.exists():
    run(sys.executable, 'scripts/prepare_gse209926_counts.py', archive,
        'config/sample_annotations/GSE209926.csv', counts)
    run('Rscript', 'R/run_deseq2_counts.R', counts,
        'config/sample_annotations/GSE209926.csv', 'GSE209926',
        'results/de/GSE209926_disease-control.csv')
else:
    print('Skipped: archive absent or RUN_GSE209926_DE=False')

## 6. GSE207890 blocked myofiber DE and animal-level sensitivity

In [ ]:
if RUN_GSE207890_BLOCKED_DE:
    run('Rscript', 'R/run_geo_microarray_blocked.R', 'GSE207890',
        'config/sample_annotations/GSE207890.csv', 'results/de')
else:
    print('Blocked myofiber stage disabled')

## 7. Cross-species, tissue-aware meta-analysis

In [ ]:
mapping = PROJECT_ROOT / 'resources/ensembl_human_mouse_one_to_one.tsv'
if RUN_META_ANALYSIS and mapping.exists():
    run('sma-xspecies', 'meta')
    # Prespecified sensitivity including the line-confounded GSE69175 study.
    run('sma-xspecies', 'meta', '--include-sensitivity-only')
else:
    print('Skipped: ortholog mapping absent or RUN_META_ANALYSIS=False')

## 8. Treatment reversal, splicing, and final audit

In [ ]:
if RUN_TREATMENT_REVERSAL:
    run('sma-xspecies', 'rescue', check=False)
if RUN_SPLICING:
    run('sma-xspecies', 'splicing', check=False)
run('pytest')
run('sma-xspecies', 'figures', check=False)
run('sma-xspecies', 'report')

## 9. Review before interpretation

Open `data/audit/manifest_validation.csv`, `results/run_record.json`, the per-study
QC outputs, and `docs/PUBLICATION_GATES.md`. A nonempty table is not automatically
valid evidence: check participating studies, species, tissue stratum, heterogeneity,
and the leave-one-dataset-out result for every prioritized gene.

The notebook intentionally does not run enrichment or a network unless versioned
gene-set and STRING bulk files are provided. This prevents silent dependence on a
changing web service or an incomplete chunked graph.